# MIDAS Bridge Model — a guided walkthrough

**Who this is for.** Bridge engineers who want to drive a MIDAS Civil NX model
from Python without writing Python. Every step below is a short, copy-able block
with plain-English notes on *what it does* and *what you should see*. You do not
need to know Python — read the comments, run the cell, read the table.

**What it shows.** Using one real model — **Brent Spence Unit 13** — we connect
to Civil NX, read the model, run the analysis, and pull engineering results back
out: member forces, a load rating per member, the controlling members, where the
deck deflects most, support reactions, and how the bridge responds to the live
load. It is the tour of the `civilpy` MIDAS library's *useful* features.

**It is non-destructive.** We only ever *open*, *read*, and *analyze in memory*.
The notebook never calls `save()`, so your source `.mcb` file is never changed.

**Before you start.**
1. Open **Civil NX** and load nothing — the notebook opens the model for you.
2. Apps ▸ API ▸ turn the API on; put your `MIDAS_API_KEY` in `~/secrets.json`.
3. Run this from the **civilpy repo root** (it imports the library source
   directly — no install needed) and point `MCB_DIR` at your model folder.


## Step 1 — Connect to Civil NX

`MidasCivil()` is your handle to the running program. `ping()` confirms the
connection is live before we do anything else.


In [ ]:
import os, sys, math, pathlib
import pandas as pd

sys.path.insert(0, os.path.abspath("."))          # run from the civilpy repo root
from src.civilpy.structural.midas import MidasCivil, parse_result_table, envelope
from src.civilpy.structural.aashto import lrfd

midas = MidasCivil()                               # base URL + key from ~/secrets.json
print(midas, "| connected:", midas.ping())

# Point this at your flattened-model folder, then at the one model we tour.
MCB_DIR = os.environ.get("MCB_DIR", r"")
MODEL   = os.path.join(MCB_DIR, "Brent Spence Unit 13.mcb")
print("Model:", MODEL)


## Step 2 — Open the model (read-only)

`open()` loads the `.mcb` into the live Civil NX session. Large models take a
moment; the library waits patiently (a long timeout) so the call doesn't give up
mid-load. We never `save`, so the file on disk is untouched.


In [ ]:
midas.open(MODEL)
print("Opened:", os.path.basename(MODEL))


## Step 3 — What's in the model?

`summarize()` counts the model's ingredients in one shot; `capability()` reads
those counts and tells you, in a sentence, whether the model can be analyzed and
load-rated. Use this to sanity-check a model *before* spending time solving it.


In [ ]:
s = midas.summarize()
can_analyze, can_rate, note = midas.capability(summary=s)

print(f"  nodes        : {s['nodes']:>6}")
print(f"  elements     : {s['elems']:>6}   by type: {s['elem_types']}")
print(f"  section types: {s['sect_types']}")
print(f"  supports     : {s['supports']:>6}")
print(f"  static loads : {s['static_loads']:>6}")
print(f"  moving loads : {s['moving_loads']:>6}")
print(f"  combinations : {s['combinations']:>6}")
print(f"\n  Can analyze? {can_analyze}    Can live-load rate? {can_rate}")
print(f"  Verdict: {note}")


## Step 4 — List the members (girders)

The model's *elements* are its physical members. We pull every element, keep the
`BEAM` members (the girders/floorbeams), and attach a readable section and
material name plus the member length. This is the member list you'd rate.


In [ ]:
elem = midas.elements().get("ELEM", {})
node = midas.nodes().get("NODE", {})
sect_names = {k: v.get("NAME", f"SECT {k}") for k, v in midas.sections().get("SECT", {}).items()}
matl_names = {k: v.get("NAME", f"MATL {k}") for k, v in midas.materials().get("MATL", {}).items()}

def member_length(e):
    ns = e.get("NODE", [])
    if len(ns) < 2 or str(ns[0]) not in node or str(ns[1]) not in node:
        return None
    a, b = node[str(ns[0])], node[str(ns[1])]
    return round(math.dist((a["X"], a["Y"], a["Z"]), (b["X"], b["Y"], b["Z"])), 2)

beams = []
for eid, e in elem.items():
    if e.get("TYPE") != "BEAM":
        continue
    beams.append({"member": int(eid),
                  "section": sect_names.get(str(e.get("SECT")), e.get("SECT")),
                  "material": matl_names.get(str(e.get("MATL")), e.get("MATL")),
                  "length_ft": member_length(e)})
members = pd.DataFrame(beams).sort_values("member").reset_index(drop=True)
print(f"{len(members)} beam members")
members.head(15)


## Step 5 — Run the analysis

`analyze()` runs the finite-element solve **in memory**. Nothing is written to
disk. A big model can take minutes; the library uses a long timeout so it isn't
cut off mid-solve.


In [ ]:
midas.analyze()
print("Analysis complete — results are now available to query.")


## Step 6 — Pull the member forces

Results are keyed by *load case name*. A plain model exposes static cases as
`Name(ST)` and moving loads as `Name(MV:all)`; a staged model keeps results only
under load **combinations** `Name(CB)`. We don't have to guess — the helper below
tries the likely case names widest-first and stops when forces come back.

`beam_forces()` returns axial, both shears, and both moments at each member end
(`Part I` and `Part J`). We flatten the response into a table with
`parse_result_table()`.


In [ ]:
ids = members["member"].tolist()

stld   = [v["NAME"] for v in midas.static_loads().get("STLD", {}).values() if v.get("NAME")][:4]
mvld   = [(v.get("LCNAME") or v.get("NAME")) for v in midas.get_db("MVLD").get("MVLD", {}).values()
          if (v.get("LCNAME") or v.get("NAME"))][:3]
combos = [v["NAME"] for v in midas.load_combinations().get("LCOM-GEN", {}).values() if v.get("NAME")][:4]

candidate_sets = [
    ("combinations + moving", [f"{c}(CB)" for c in combos] + [f"{m}(MV:all)" for m in mvld]),
    ("static + moving",       [f"{s}(ST)" for s in stld]   + [f"{m}(MV:all)" for m in mvld]),
    ("combinations only",     [f"{c}(CB)" for c in combos]),
]

force_rows, used_cases, used_case_list = [], None, []
for label, cases in candidate_sets:
    if not cases:
        continue
    rows = parse_result_table(midas.beam_forces(ids, cases))
    if rows:
        force_rows, used_cases, used_case_list = rows, label, cases
        break

print(f"Pulled {len(force_rows)} force rows using the '{used_cases}' load cases.")
forces = pd.DataFrame(force_rows)
forces.head(8)


## Step 7 — Controlling force per member

Each member has several force rows (two ends × several load cases). For rating we
want the **largest magnitude** of each force on each member — its *envelope*. We
group by member and take the peak moment, shear, and axial. Sorting by moment
shows the most heavily loaded girders first.


In [ ]:
# Force columns arrive as text; convert the three we envelope to numbers.
for col in ("Moment-y", "Shear-z", "Axial"):
    forces[col + "_abs"] = pd.to_numeric(forces[col], errors="coerce").abs()

env = (forces.groupby("Elem")
       .agg(max_moment_kft=("Moment-y_abs", "max"),
            max_shear_kip =("Shear-z_abs",  "max"),
            max_axial_kip =("Axial_abs",    "max"))
       .reset_index().rename(columns={"Elem": "member"}))
env["member"] = pd.to_numeric(env["member"], errors="coerce")
env = env.sort_values("max_moment_kft", ascending=False).reset_index(drop=True)

top = env.iloc[0]
print(f"Highest moment: member {int(top['member'])} at {top['max_moment_kft']:.0f} kip-ft")
env.head(10)


## Step 8 — Load rating per member (AASHTO MBE §6A.4.2.1)

The rating factor is `RF = (C − γ_DC·DC − γ_DW·DW) / (γ_LL·LL(1+IM))` — capacity
left over for live load, divided by the factored live-load demand. `RF ≥ 1.0`
passes at the inventory level.

`lrfd.rating_factor()` is civilpy's implementation. It needs the member
**capacity** `C` (here we use one representative `Mn` so the mechanics are clear —
in production you'd compute each member's `Mn` from its section with civilpy's
steel/concrete capacity checks) and the unfactored `DC`, `DW`, and live-load
effects. The member with the **lowest RF controls** the rating.


In [ ]:
# Representative capacity + permanent-load split (replace with per-member values).
Mn_capacity = 9000.0      # kip-ft nominal moment capacity (per girder)
DC_FRACTION = 0.55        # share of the envelope moment that is dead load (DC)
DW_FRACTION = 0.10        # wearing-surface share (DW)

def rate_member(moment):
    dc = moment * DC_FRACTION
    dw = moment * DW_FRACTION
    ll = moment * (1 - DC_FRACTION - DW_FRACTION)     # live-load share
    r = lrfd.rating_factor(Mn_capacity, dc, ll * 1.33, dw=dw)   # 1.33 = 1 + IM
    return round(r.capacity, 2)

rating = env.copy()
rating["RF_inventory"] = rating["max_moment_kft"].apply(rate_member)
rating = rating.sort_values("RF_inventory").reset_index(drop=True)

worst = rating.iloc[0]
verdict = "PASSES" if worst["RF_inventory"] >= 1.0 else "is BELOW 1.0 — investigate"
print(f"Controlling member: {int(worst['member'])}  RF = {worst['RF_inventory']:.2f}  ({verdict})")
rating[["member", "max_moment_kft", "RF_inventory"]].head(10)


## Step 9 — Deflections: how much, and where

Deflections come from the same results, via the *displacement* table. We pull
the global displacements, flatten them, and find the largest vertical movement
(`DZ`) and the node it occurs at — then look that node up in the geometry to
report *where* on the bridge it happens.

> The first line prints the actual column names the model returned. If your model
> labels the vertical component differently, change `VERT` to match.


In [ ]:
disp_rows = parse_result_table(midas.result_table(
    "Displacement", table_type="DISPLACEMENTG",
    components=["Node", "Load", "DX", "DY", "DZ"],
    node_elems={"KEYS": [int(n) for n in node][:2000]},
    load_case_names=used_case_list))

disp = pd.DataFrame(disp_rows)
print("Displacement table columns:", list(disp.columns))

VERT = "DZ"
disp["_dz"] = pd.to_numeric(disp.get(VERT), errors="coerce").abs()
worst = disp.loc[disp["_dz"].idxmax()]
nid = str(worst.get("Node", "")).strip()
loc = node.get(nid, {})
print(f"\nMax vertical deflection: {worst['_dz']:.3f} in  at node {nid}")
if loc:
    print(f"  located at X={loc['X']:.1f}, Y={loc['Y']:.1f}, Z={loc['Z']:.1f} (model units)")
disp.sort_values("_dz", ascending=False).head(5)


## Step 10 — Support reactions

The reaction table gives the forces the bridge delivers to its bearings — useful
for bearing and substructure checks. Same pattern: query the table, flatten it,
read it.


In [ ]:
rxn_rows = parse_result_table(midas.result_table(
    "Reaction", table_type="REACTIONG",
    components=["Node", "Load", "FX", "FY", "FZ"],
    node_elems={"KEYS": [int(n) for n in node][:2000]},
    load_case_names=used_case_list))
reactions = pd.DataFrame(rxn_rows)
if not reactions.empty:
    reactions["_fz"] = pd.to_numeric(reactions.get("FZ"), errors="coerce").abs()
    print(f"Largest vertical reaction: {reactions['_fz'].max():.0f} kip")
reactions.head(8)


## Step 11 — Live load: how the bridge responds to the design vehicle

A load rating is only as good as the truck it's rated for. The moving-load setup
lives in three tables you can read directly:

- `MVCD` — the moving-load **code** (e.g. *AASHTO LRFD* → HL-93).
- `MVHL` — the **vehicles** (HL-93, HS20, or a custom permit truck).
- `MVLD` — the moving-load **cases** that apply those vehicles to lanes.

Reading them tells you exactly what live load the results above represent.


In [ ]:
mvcd = midas.get_db("MVCD").get("MVCD", {})
mvhl = midas.get_db("MVHL").get("MVHL", {})
mvld = midas.get_db("MVLD").get("MVLD", {})

print("Moving-load code :", {k: v.get("CODE", v) for k, v in mvcd.items()} or "(none)")
print("Vehicles (MVHL)  :", [v.get("NAME", k) for k, v in mvhl.items()] or "(none)")
print("Moving cases     :", [(v.get("LCNAME") or v.get("NAME")) for v in mvld.values()] or "(none)")


### Swapping in a different vehicle (HS20 / HL-93 / permit)

To see how the bridge reacts to a *different* truck, you read one existing vehicle
as a **template**, change what you need (the standard vehicle code or the axle
loads/spacings for a permit truck), write it back with `put_db("MVHL", ...)`, then
re-run Steps 5–8. Because we never `save()`, this only changes the **in-memory**
model — your file stays as it was.

The block below is a template, left un-run so the walkthrough doesn't alter your
live model unasked. Fill in the vehicle fields from the `MVHL` row you printed
above, then remove the `if False:` guard to apply it.


In [ ]:
if False:   # flip to True to apply, then re-run Steps 5-8 to compare RFs
    template_id = next(iter(mvhl))                 # an existing vehicle as the shape
    new_vehicle = dict(mvhl[template_id])          # copy it
    new_vehicle["NAME"] = "HS20-44"                # <- standard code, or edit axles
    midas.put_db("MVHL", {"99": new_vehicle})      # write into the in-memory model
    midas.analyze()                                # re-solve
    # ... then repeat Step 6 (beam_forces) and Step 8 (rate_member) and compare.
print("Template ready — see the comments to apply a new vehicle.")


## Cheat sheet — the methods you used

| Call | What it gives you |
|---|---|
| `MidasCivil()` / `.ping()` | connect to Civil NX, confirm it's live |
| `.open(path)` | load a model (read-only — never saved) |
| `.summarize()` / `.capability()` | model contents + can-it-be-rated verdict |
| `.elements()` / `.nodes()` / `.sections()` / `.materials()` | model geometry & properties |
| `.analyze()` | run the solve in memory |
| `.beam_forces(ids, cases)` | member end forces (axial, shear, moment) |
| `.result_table("Displacement"/"Reaction", …)` | any other results table |
| `parse_result_table(resp)` | turn a MIDAS table into rows you can tabulate |
| `lrfd.rating_factor(C, dc, ll_im, dw=…)` | AASHTO MBE §6A.4.2.1 rating factor |
| `.get_db("MVCD"/"MVHL"/"MVLD")` / `.put_db(...)` | read / change the moving-load setup |

Everything here is read-or-analyze only. To make changes permanent you would call
`.save()` — deliberately **not** used in this walkthrough.
